# 1. 理解工具

In [11]:
from langchain.tools import tool

@tool   # 装饰器
def calcutor(op:str, p1:float, p2:float)->float:
    '''
    执行算术运算，包含加法，减法，乘法，除法，求余，以运算优先级符号。
    '''
    return 100

In [12]:
print(type(calcutor))

<class 'langchain_core.tools.structured.StructuredTool'>


In [13]:
calcutor.description
calcutor.func
calcutor.name
calcutor.args_schema

langchain_core.utils.pydantic.calcutor

- 工具与模型的关系

In [15]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="ollama:qwen3:4b")
print(type(model))
model_with_tools = model.bind_tools([calcutor])
print(model_with_tools)

<class 'langchain_ollama.chat_models.ChatOllama'>
bound=ChatOllama(model='qwen3:4b') kwargs={'tools': [{'type': 'function', 'function': {'name': 'calcutor', 'description': '执行算术运算，包含加法，减法，乘法，除法，求余，以运算优先级符号。', 'parameters': {'properties': {'op': {'type': 'string'}, 'p1': {'type': 'number'}, 'p2': {'type': 'number'}}, 'required': ['op', 'p1', 'p2'], 'type': 'object'}}}]} config={} config_factories=[]


In [16]:
print(type(model_with_tools))

<class 'langchain_core.runnables.base.RunnableBinding'>


In [17]:
response = model_with_tools.invoke("计算45与55的和")
print(response)
print(type(response))

content='' additional_kwargs={} response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-09T06:50:02.8071329Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14761556200, 'load_duration': 8122707200, 'prompt_eval_count': 178, 'prompt_eval_duration': 70052800, 'eval_count': 436, 'eval_duration': 6423729500, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'} id='lc_run--019d7101-0f09-76a2-be0d-0593c356c1db-0' tool_calls=[{'name': 'calcutor', 'args': {'p2': 55, 'op': '+', 'p1': 45}, 'id': '0e629860-e20e-4a69-b416-68db469d349f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 178, 'output_tokens': 436, 'total_tokens': 614}
<class 'langchain_core.messages.ai.AIMessage'>


In [23]:
response = model_with_tools.invoke("56与67的积是多少?")
# print(response)
if response.tool_calls:
    print("继续调用工具：", response.tool_calls)
else:
    print("直接结果:", response.text)

继续调用工具： [{'name': 'calcutor', 'args': {'op': '*', 'p1': 56, 'p2': 67}, 'id': '77855cd8-6153-4e11-9fd3-3580df1f0058', 'type': 'tool_call'}]


In [25]:
calcutor.invoke({"op":"+", "p1":34, "p2":78})

100

In [40]:
from langchain_core.tools.structured import StructuredTool
from pydantic import BaseModel, Field   # 定义参数模式schema
class InfoSchema(BaseModel):
    x:int = Field(description="订单号")
    y:str = Field(description="订单要求")

def get_info(x:int, y:str)->str:
    return "hello"

t1 = StructuredTool(   # .from_function(   # 工具
    func=get_info,
    name="获取客户信息",    # 默认情况下与函数名相同
    description="获取客户信息，返回客户的订单信息",
    args_schema=InfoSchema
)
print(t1.invoke({"x":100, "y":"要求高质量"}))

hello


In [43]:
@tool("查询天气", 
      description="查询天气"
      "，需要一个字符串格式的日期参数")
def get_weather(a):
    return "Hello"
print(get_weather.name)
print(get_weather.description)
print(get_weather.func)
print(get_weather.args_schema)

查询天气
查询天气，需要一个字符串格式的日期参数
<function get_weather at 0x000001965A303C40>
<class 'langchain_core.utils.pydantic.查询天气'>


- 工具的保留参数

In [44]:
from langchain.tools import ToolRuntime
from langchain_core.runnables import RunnableConfig
@tool("访问用户的数据")
def get_message(runtime:ToolRuntime, config:RunnableConfig)->str:  # runtime与config是智能体传递过来：全局
    """
    获取用户的存储的数据，以及最近用户访问的数据。
    """
    print("*" * 80)
    print(runtime.state)
    print("*" * 80)
    print(runtime.context)
    runtime.context={}
    runtime.context["data"]="hello"
    print("*" * 80)
    print(runtime.store)
    print("*" * 80)
    print(config)
    print("*" * 80)
    return "你好"

In [45]:
# 创建模型（省略）
# 创建智能体
from langchain.agents import create_agent
agent = create_agent(
    model="ollama:qwen3:4b",
    tools=[get_message],
    system_prompt="你是一个数据专家,专门处理用户数据"
)
# 调用


In [46]:
response = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content":"请给我查询用户存储的历史数据"
        }
    ]
})
print(response["messages"][-1].text)

********************************************************************************
{'messages': [HumanMessage(content='请给我查询用户存储的历史数据', additional_kwargs={}, response_metadata={}, id='7fc5ac92-f67f-4908-b5b9-440b8324c3f2'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-09T08:04:32.9135621Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9204037000, 'load_duration': 1704586600, 'prompt_eval_count': 145, 'prompt_eval_duration': 114288200, 'eval_count': 428, 'eval_duration': 7225631800, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019d7145-5a1b-7903-9b04-753f8308b2c1-0', tool_calls=[{'name': '访问用户的数据', 'args': {}, 'id': '0d7e4cbf-86f5-404f-afc7-1b5abeb94155', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 428, 'total_tokens': 573})]}
********************************************************************************
None
************

----

# 2. 理解智能体

## 2.1. 中间件与动态模型切换

In [49]:
from langchain.agents import create_agent    # 智能体创建
from langchain.chat_models import init_chat_model 
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse   # 调用模型前，必须调用中间件(拦截器)
from langchain.tools import tool
from langchain.messages import SystemMessage
# 创建两个模型(不要使用构造器：init_chat_model在完成对象构建，同时还对某些数据进行了初始化)
model_4b = init_chat_model(model="ollama:qwen3:4b", num_predict=1024, temperature=0.1)
model_9b = init_chat_model(model="ollama:qwen3.5:9b", num_predict=512, temperature=0.7)

# 中间件wrap_model_call 这个装饰器：模型调用前，先调用中间件（可以实现模型切换）
@wrap_model_call
def switch_model(request:ModelRequest, handler)->ModelResponse:
    # 切换我们的模型
    # 假设一个条件（聊天的信息）
    if request.messages[0].content.startswith("#"):  # 用户的问题是#开头，使用编码模型，否则就是问答QA模型
        # 切换到编码模型（模型支持tool）
        print("模型切换到9b")
        model = model_9b
    else:
        # 切换到问答模型
        print("模型切换到4b")
        model = model_4b
    return handler(request.override(model=model))

agent = create_agent(
    model=model_4b,
    tools=[], 
    middleware=[switch_model],
    system_prompt=SystemMessage(content="你是一个友好的助手。")   # 规范
)

In [50]:
agent.invoke({
    "messages":[
        {
            "role": "user",
            "content": "#使用C++写一个冒泡算法"
        }
    ]
})

模型切换到9b


{'messages': [HumanMessage(content='#使用C++写一个冒泡算法', additional_kwargs={}, response_metadata={}, id='285e0542-080e-41a6-bcc3-28e32611ebb2'),
  AIMessage(content='# C++ 冒泡排序算法实现\n\n冒泡排序（Bubble Sort）是一种简单直观的排序算法，适用于学习排序原理。以下是完整的 C++ 实现，包含详细注释、测试示例和性能说明。\n\n---\n\n## ✅ 基础版本\n\n```cpp\n#include <iostream>\nusing namespace std;\n\nvoid bubbleSort(int arr[], int n) {\n    for (int i = 0; i < n - 1; i++) {\n        // 每轮将当前未排序部分的最大值"冒泡"到末尾\n        for (int j = 0; j < n - i - 1; j++) {\n            if (arr[j] > arr[j + 1]) {\n                // 交换相邻元素\n                int temp = arr[j];\n                arr[j] = arr[j + 1];\n                arr[j + 1] = temp;\n            }\n        }\n    }\n}\n\nint main() {\n    int arr[] = {64, 34, 25, 12, 22, 11, 90};\n    int n = sizeof(arr) / sizeof(arr[0]);\n\n    cout << "排序前: ";\n    for (int i = 0; i < n; i++)\n        cout << arr[i] << " ";\n    cout << endl;\n\n    bubbleSort(arr, n);\n\n    cout << "排序后: ";\n    for (int i = 0; i < n; i++)\n     

In [51]:
agent.invoke({
    "messages":[
        {
            "role": "user",
            "content": "你是谁?"
        }
    ]
})

模型切换到4b


{'messages': [HumanMessage(content='你是谁?', additional_kwargs={}, response_metadata={}, id='63f85359-3b61-400a-ba7c-9b4d8602c421'),
  AIMessage(content='你好！我是通义千问（Qwen），阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我可以帮助你回答问题、创作文字（比如写故事、写公文、写邮件、写剧本等）、进行逻辑推理、编程，甚至表达观点。有什么我可以帮你的吗？ 😊', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-09T08:59:41.4200947Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9418734300, 'load_duration': 2814243500, 'prompt_eval_count': 24, 'prompt_eval_duration': 26423200, 'eval_count': 438, 'eval_duration': 6414646800, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019d7177-d51e-7f60-bb98-e18a379baf80-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 438, 'total_tokens': 462})]}

## 2.2. 智能体的输出格式

- 不仅仅是数据的输出格式，而是智能体的能力
    - 格式输出能力
    - 工具的执行能力

In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

class ContractInfo(BaseModel):
    # 定义格式
    name  : str = Field(description="姓名")
    email : str = Field(description="电子邮箱")
    phone : str = Field(description="电话号码")

agent = create_agent(
    model="ollama:qwen3:4b",
    tools=[],
    middleware=[],
    system_prompt=SystemMessage(content="我是一个信息提取助手，对用户输入的信息提取联系人的信息"),
    response_format=ContractInfo
)

state = {"messages":[]}

conversation = [
    "我是杨强。",
    "我的邮箱是38395870@qq.com。",
    "我的电话是13338629985。",
    "前面的信息都记录了吗？",
    "帮我整理一下完整的联系人信息。"
]

for query in  conversation:
    print(F"----------------用户输入:{query}-----------------")
    state["messages"].append(HumanMessage(content=query))    # 推荐
    # state["messages"].append({"role":"user", "content":query})
    outputs = agent.invoke(state)
    print(outputs["messages"][-1].text)

----------------用户输入:我是杨强。-----------------
Returning structured response: name='杨强' email='' phone=''
----------------用户输入:我的邮箱是38395870@qq.com。-----------------
Returning structured response: name='杨强' email='38395870@qq.com' phone=''
----------------用户输入:我的电话是13338629985。-----------------
Returning structured response: name='杨强' email='38395870@qq.com' phone='13338629985'
----------------用户输入:前面的信息都记录了吗？-----------------
Returning structured response: name='杨强' email='38395870@qq.com' phone='13338629985'
----------------用户输入:帮我整理一下完整的联系人信息。-----------------
Returning structured response: name='杨强' email='38395870@qq.com' phone='13338629985'
